# 🧠 手書き数字認識モデル ファインチューニング & TFJS変換 ノートブック

このノートブックは、手書き数字収集ツール (`collect_single.html`) で集めた苦手な数字データ (`.jsonl`) と、ベースモデル (`mnist.keras`) を使って**ファインチューニング（追加学習）**を行い、Webアプリ用の **TensorFlow.js 形式 (`model.json` + `.bin`)** に自動変換してダウンロードします。

### ⚡ 使い方（3ステップ）
1. メニューの **「ランタイム」 >「すべてのセルを実行」** をクリックします。
2. ファイル選択ボタンが表示されたら、`mnist.keras` と `mnist_finetune_XXXX.jsonl` をアップロードします。
3. 学習完了後、変換された `model_updated.zip` が自動でダウンロードされます！

## 1. 必要なライブラリのインストール

In [ ]:
!pip install -q tensorflowjs
import os
import glob
import json
import shutil
import numpy as np
import tensorflow as tf
from google.colab import files
import matplotlib.pyplot as plt

print(f"TensorFlow Version: {tf.__version__}")

## 2. ファイルの準備（アップロード）
`mnist.keras` と、ダウンロードした `mnist_finetune_XXXX.jsonl` をアップロードしてください。

In [ ]:
# すでにアップロードされているか確認
keras_files = glob.glob("*.keras")
jsonl_files = glob.glob("*.jsonl")

if not keras_files or not jsonl_files:
    print("📂 'mnist.keras' と 'dataset.jsonl' を選択してアップロードしてください...")
    uploaded = files.upload()
    keras_files = glob.glob("*.keras")
    jsonl_files = glob.glob("*.jsonl")

assert len(keras_files) > 0, "⚠️ .keras ファイルが見つかりません。mnist.keras をアップロードしてください。"
assert len(jsonl_files) > 0, "⚠️ .jsonl ファイルが見つかりません。収集した jsonl ファイルをアップロードしてください。"

keras_model_path = keras_files[0]
jsonl_path = jsonl_files[0]
print(f"✅ モデルファイル: {keras_model_path}")
print(f"✅ データセット: {jsonl_path}")

## 3. モデルの読み込み

In [ ]:
model = tf.keras.models.load_model(keras_model_path)
print("モデル構造:")
model.summary()

## 4. 収集データの読み込み & 破滅的忘却防止ブレンド
集めた「苦手な数字」だけを学習させると他の数字を忘れてしまうため、元の MNIST データセットとブレンドし、さらにデータ拡張（回転・平行移動）を行って学習データを強固にします。

In [ ]:
# 収集データのロード
custom_images = []
custom_labels = []

with open(jsonl_path, 'r', encoding='utf-8') as f:
    for line in f:
        if not line.strip():
            continue
        item = json.loads(line)
        pixels = np.array(item['pixels'], dtype=np.float32).reshape(28, 28, 1)
        label = int(item['label'])
        custom_images.append(pixels)
        custom_labels.append(label)

custom_images = np.array(custom_images)
custom_labels = np.array(custom_labels)
print(f"📊 収集データ件数: {len(custom_images)} 件")

# 元のMNISTデータセットをロード
(x_mnist, y_mnist), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_mnist = (x_mnist / 255.0).astype(np.float32)[..., np.newaxis]
x_test = (x_test / 255.0).astype(np.float32)[..., np.newaxis]

# 収集データのデータ拡張（Augmentation）: 回転・平行移動・ズーム
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.04, fill_mode='constant', fill_value=0.0), # 約±14度
    tf.keras.layers.RandomTranslation(0.06, 0.06, fill_mode='constant', fill_value=0.0), # 約±1.7px
    tf.keras.layers.RandomZoom((-0.08, 0.08), fill_mode='constant', fill_value=0.0)
])

# 収集データを各サンプル50倍に水増し
augmented_list = []
augmented_labels = []
MULTIPLIER = 50

for img, lbl in zip(custom_images, custom_labels):
    batch = np.repeat(img[np.newaxis, ...], MULTIPLIER, axis=0)
    aug_batch = data_augmentation(batch, training=True).numpy()
    augmented_list.append(aug_batch)
    augmented_labels.extend([lbl] * MULTIPLIER)

if augmented_list:
    aug_x = np.concatenate(augmented_list, axis=0)
    aug_y = np.array(augmented_labels)
else:
    aug_x = custom_images
    aug_y = custom_labels

# MNISTから標準データをバランスよくサンプリング（水増しデータの3〜4倍程度）
mnist_sample_count = max(3000, len(aug_x) * 3)
indices = np.random.choice(len(x_mnist), min(mnist_sample_count, len(x_mnist)), replace=False)
blend_mnist_x = x_mnist[indices]
blend_mnist_y = y_mnist[indices]

# データセットの統合とシャッフル
x_train = np.concatenate([blend_mnist_x, aug_x], axis=0)
y_train = np.concatenate([blend_mnist_y, aug_y], axis=0)

perm = np.random.permutation(len(x_train))
x_train = x_train[perm]
y_train = y_train[perm]

print(f"🚀 統合学習データ総数: {len(x_train)} 枚 (MNIST: {len(blend_mnist_x)}, 拡張収集データ: {len(aug_x)})")

## 5. ファインチューニング実行
既存の重みを壊さないよう、小さな学習率 (`lr=0.0001`) で微調整（ファインチューニング）します。

In [ ]:
# 低学習率でコンパイル
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# ファインチューニング（5エポック）
history = model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=64,
    validation_data=(x_test, y_test)
)

# 収集データに対する最終認識率の確認
custom_preds = np.argmax(model.predict(custom_images), axis=1)
custom_acc = np.mean(custom_preds == custom_labels) * 100
print(f"\n🎯 収集した苦手データに対する新モデルの認識率: {custom_acc:.1f}%")

## 6. TensorFlow.js 形式への変換 ＆ ダウンロード
学習済みのモデルを TensorFlow.js 形式 (`model.json` + `group1-shard1of1.bin`) に変換し、zip ファイルにしてダウンロードします。

In [ ]:
out_dir = "model_tfjs"
if os.path.exists(out_dir):
    shutil.rmtree(out_dir)
os.makedirs(out_dir, exist_ok=True)

# 一時的に Keras 形式で保存
temp_keras = "temp_finetuned.keras"
model.save(temp_keras)

# TFJS形式へ変換
!tensorflowjs_converter --input_format=keras {temp_keras} {out_dir}

# zip化
zip_name = "model_updated"
shutil.make_archive(zip_name, 'zip', out_dir)
print(f"📦 生成完了: {zip_name}.zip")

# ダウンロード実行
files.download(f"{zip_name}.zip")
print("🎉 ダウンロードが開始されました！")
print("この zip を展開し、中身（model.json と .bin）をアプリの model/ フォルダに上書きしてください。")